Bengaluru House Prices

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


data = pd.read_csv("/content/sample_data/bengaluru_house_prices.csv")

print(data.head())
print(data.shape)

# Remove duplicated records
data = data.drop_duplicates().reset_index(drop=True)

# Extract the number of bedrooms
data["bedrooms"] = (
    data["size"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype(float)
)

# Convert total_sqft values such as "1200-1500" to their average
def clean_area(value):
    value = str(value)

    if "-" in value:
        parts = value.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return np.nan

    try:
        return float(value)
    except ValueError:
        return np.nan


data["total_sqft"] = data["total_sqft"].apply(clean_area)

# Remove rows where the target or main area value is missing
data = data.dropna(subset=["total_sqft", "price"])

# Remove unrealistic values
data = data[(data["total_sqft"] >= 200) & (data["price"] > 0)]

print(data.head())


features = [
    "area_type",
    "availability",
    "location",
    "bedrooms",
    "total_sqft",
    "bath",
    "balcony"
]

X = data[features]
y = data["price"]

categorical_features = [
    "area_type",
    "availability",
    "location"
]

numerical_features = [
    "bedrooms",
    "total_sqft",
    "bath",
    "balcony"
]

preprocess = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("fill", SimpleImputer(strategy="most_frequent")),
                ("encode", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "numerical",
            SimpleImputer(strategy="median"),
            numerical_features
        )
    ]
)




X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=10
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])



regression_model = Pipeline([
    ("preprocessing", preprocess),
    (
        "forest",
        RandomForestRegressor(
            random_state=10
        )
    )
])

search_params = {
    "forest__n_estimators": [150, 250],
    "forest__max_depth": [12, 18],
    "forest__min_samples_split": [2, 4]
}

reg_grid = GridSearchCV(
    estimator=regression_model,
    param_grid=search_params,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

reg_grid.fit(X_train, y_train)

print("Optimal Parameters:")
print(reg_grid.best_params_)




house_predictions = reg_grid.predict(X_test)

mae = mean_absolute_error(y_test, house_predictions)
rmse = np.sqrt(mean_squared_error(y_test, house_predictions))
r2 = r2_score(y_test, house_predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

Churn Modelling

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
)



churn_data = pd.read_csv("/content/sample_data/Churn_Modelling.csv")

print(churn_data.head())
print(churn_data.shape)



# Delete duplicated rows
churn_data = churn_data.drop_duplicates().reset_index(drop=True)

# Remove columns that are not useful for prediction
X = churn_data.drop(
    columns=["Exited", "RowNumber", "CustomerId", "Surname"]
)

y = churn_data["Exited"]

categorical_cols = ["Geography", "Gender"]

numeric_cols = [
    column for column in X.columns
    if column not in categorical_cols
]

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numeric_cols)




data_processor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("missing", SimpleImputer(strategy="most_frequent")),
                ("one_hot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_cols
        ),
        (
            "numerical",
            SimpleImputer(strategy="median"),
            numeric_cols
        )
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=10,
    stratify=y
)



classifier_pipeline = Pipeline([
    ("processor", data_processor),
    (
        "classifier",
        RandomForestClassifier(
            random_state=10
        )
    )
])

parameters = {
    "classifier__n_estimators": [150, 250],
    "classifier__max_depth": [15, 20],
    "classifier__min_samples_split": [2, 5]
}

clf_grid = GridSearchCV(
    estimator=classifier_pipeline,
    param_grid=parameters,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

clf_grid.fit(X_train, y_train)

print("Best Parameters:")
print(clf_grid.best_params_)




churn_pred = clf_grid.predict(X_test)
churn_probability = clf_grid.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, churn_pred)
roc_auc = roc_auc_score(y_test, churn_probability)

print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)

print("\nClassification Report:")
print(classification_report(y_test, churn_pred))